# 当前软件的 das_veh 轨迹拾取：服务器参数实验

本 Notebook 内嵌当前工作区代码快照，目标是复现 **软件现在实际使用的实现**，并查看参数如何影响峰、起点和轨迹。
完整复制 `das_veh_tracking.py`、`tracking.py`、`car_tracking_utils.py`、`bin_reader.py`、`data_group.py`；
从另两个模块原样复制运行所需的定义。原始相对导入、函数体和默认参数保持原样。
`%%copied_module` 把源码加载到内存包中，不写出 Python 文件，也不需要软件工程或 Qt。
新增单元只负责数据配置、调用现有函数、记录中间结果、画图和对比实验。

**默认条件：1000 Hz、原始通道 290–520、0.01–1 Hz、0–120 km/h、双向、正负峰分别检测。**
全部滤波、找峰、Gaussian 得分和跟踪保留原始时间采样点，不进行时间或空间降采样，不读取人工标注。
图像由 Matplotlib 显示到屏幕像素，传入图像与波形的数组仍是全采样率数据。

使用顺序：
1. 在服务器环境安装依赖，选择 Python 3 内核。
2. 修改“数据与参数配置”中的 BIN 路径、源内时间范围、通道间距。
3. 从上到下运行；第一次先看单通道峰与起点，再看轨迹。
4. 修改 `PARAMETERS` 后，重跑“运行一次”及下面的诊断单元；批量实验需显式开启。

安装命令（在服务器终端中运行）：
```bash
python -m pip install numpy scipy matplotlib jupyterlab ipykernel
```
源码区可折叠浏览，但第一次必须执行。Notebook 本身包含完整运行依赖源码，不需要复制仓库。
真实 BIN 路径由你在服务器填写；文件按列表顺序连续拼接，当前实现不从文件日期自动补空隙。

In [ ]:
import sys
import types
import json
import hashlib
import platform
import time
from pathlib import Path

import numpy as np
import scipy
import matplotlib
import matplotlib.pyplot as plt

SOURCE_MANIFEST = {'_das_veh_lab_snapshot.utils.bin_reader': {'path': 'utils/bin_reader.py', 'copy_kind': 'whole_module', 'line_ranges': [[1, 141]], 'file_bytes_sha256': '8df450c3cf9f4b5a9beaa9adaa10ccda8862bc6f14e01744cd7c2b373c93eee0', 'embedded_sha256': '8df450c3cf9f4b5a9beaa9adaa10ccda8862bc6f14e01744cd7c2b373c93eee0'}, '_das_veh_lab_snapshot.utils.classes.data_group': {'path': 'utils/classes/data_group.py', 'copy_kind': 'whole_module', 'line_ranges': [[1, 172]], 'file_bytes_sha256': 'a14309418556635dd5f8388c75e686a2113a9dc4153407a25d47f4a30d8c4521', 'embedded_sha256': 'a14309418556635dd5f8388c75e686a2113a9dc4153407a25d47f4a30d8c4521'}, '_das_veh_lab_snapshot.utils.classes.video_trajectory': {'path': 'utils/classes/video_trajectory.py', 'copy_kind': 'exact_definitions', 'line_ranges': [[3, 3], [6, 6], [8, 8], [11, 11], [18, 18], [50, 86]], 'file_bytes_sha256': '1156e7ec139ce42f529e9d81ad6d500fb0225e5141bffe908761cf236b8f9952', 'embedded_sha256': '873919af0b86e62e158815558f3e0d31d12dc6b31bc152c17da1488d412244cd'}, '_das_veh_lab_snapshot.utils.classes.trajectory_prescreen': {'path': 'utils/classes/trajectory_prescreen.py', 'copy_kind': 'exact_definitions', 'line_ranges': [[8, 8], [11, 11], [28, 29], [32, 34], [37, 44], [47, 50], [53, 75]], 'file_bytes_sha256': '2b72a18f44d9ba481eef81b93f13a18ffe71a9bf809c237b2f6ea05bfefa4595', 'embedded_sha256': '102d9fe7bf5ffa81c16cca785630c7c0d86cd0f501bc6b5c015fe02400c99cb6'}, '_das_veh_lab_snapshot.utils.vendor.das_veh.car_tracking_utils': {'path': 'utils/vendor/das_veh/car_tracking_utils.py', 'copy_kind': 'whole_module', 'line_ranges': [[1, 93]], 'file_bytes_sha256': 'af73ca924b171a562b8d8e4e9e6f803e71d061094991731cef03a66990ef0e65', 'embedded_sha256': 'ed6d4883c1c033949ed45e3378f58acba275228e71f60cb0635909a23b7b182a'}, '_das_veh_lab_snapshot.utils.vendor.das_veh.tracking': {'path': 'utils/vendor/das_veh/tracking.py', 'copy_kind': 'whole_module', 'line_ranges': [[1, 267]], 'file_bytes_sha256': 'e44ce9093de3dbdaff32e1562a0856d2840752ef2a2a66f285c5c1997b94ccce', 'embedded_sha256': '780ee00e7d5159b47bbf78163b329c15cf84527c9ca84651f2e0cc6eeef461b3'}, '_das_veh_lab_snapshot.utils.classes.das_veh_tracking': {'path': 'utils/classes/das_veh_tracking.py', 'copy_kind': 'whole_module', 'line_ranges': [[1, 267]], 'file_bytes_sha256': '604a9d31209939c87559022cbe85da3a0024a06133db39416bcc7c1e6ccffc1e', 'embedded_sha256': '604a9d31209939c87559022cbe85da3a0024a06133db39416bcc7c1e6ccffc1e'}}
RUNTIME_VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__,
                    'scipy': scipy.__version__, 'matplotlib': matplotlib.__version__}

def _canonical_source(source):
    return source.replace('\r\n', '\n').rstrip('\n') + '\n'

def _ensure_package(name):
    if name in sys.modules:
        return sys.modules[name]
    parent, _, child = name.rpartition('.')
    if parent:
        _ensure_package(parent)
    module = types.ModuleType(name)
    module.__path__ = []
    module.__package__ = name
    sys.modules[name] = module
    if parent:
        setattr(sys.modules[parent], child, module)
    return module

def install_copied_module(name, source):
    expected = SOURCE_MANIFEST[name]
    digest = hashlib.sha256(_canonical_source(source).encode('utf-8')).hexdigest()
    if digest != expected['embedded_sha256']:
        raise ValueError(f"{expected['path']} 内容与复制快照不一致；请恢复该源码单元。")
    parent, _, child = name.rpartition('.')
    _ensure_package(parent)
    module = types.ModuleType(name)
    module.__package__ = parent
    module.__file__ = f"<copied:{expected['path']}>"
    sys.modules[name] = module
    setattr(sys.modules[parent], child, module)
    exec(compile(source, module.__file__, 'exec'), module.__dict__)
    # Convenience aliases for experiment cells; original imports resolve in the in-memory package.
    aliases = {'bin_reader': 'bin_reader', 'data_group': 'data_group',
               'trajectory_prescreen': 'prescreen', 'car_tracking_utils': 'tracking_utils',
               'tracking': 'tracking', 'das_veh_tracking': 'adapter'}
    if child in aliases:
        globals()[aliases[child]] = module
    print(f"已校验并加载 {expected['path']} ({expected['copy_kind']})")
    return None

if 'get_ipython' in globals() and get_ipython() is not None:
    get_ipython().register_magic_function(
        lambda line, cell: install_copied_module(line.strip(), cell),
        magic_kind='cell', magic_name='copied_module')

print('运行环境：', RUNTIME_VERSIONS)
print('源码模块：', len(SOURCE_MANIFEST), '；源码只加载到内存，不写出临时文件。')


## A. 原样复制的运行代码

阅读参数实验时可先跳到 B。以下代码单元保留当前实现；每个单元运行时校验其源码内容。
支持定义仅复制本算法调用的部分，不加载旧的降采样算法或 GUI；所复制定义没有重写。

**A1. BIN 读取和数据组：整文件复制。** 读取格式与软件一致：20 个 float32 头，数据体按通道连续存储。

In [ ]:
%%copied_module _das_veh_lab_snapshot.utils.bin_reader
import os
import numpy as np

HEADER_FLOATS = 20
HEADER_BYTES = HEADER_FLOATS * 4
HEADER_FRAMES_INDEX = 7
HEADER_CHANNELS_INDEX = 8
HEADER_SAMPLING_RATE_INDEX = 9


def bin_header_metadata(header):
    """解析 notebook 定义的 BIN 头字段。"""
    if len(header) < HEADER_FLOATS:
        raise ValueError(f"BIN 文件头长度不足，期望 {HEADER_FLOATS} 个 float32。")

    def integer_field(index, name):
        value = float(header[index])
        if not np.isfinite(value) or value <= 0:
            raise ValueError(f"BIN 文件头 {name} 无效: {value}")
        integer_value = int(value)
        if value != integer_value:
            raise ValueError(f"BIN 文件头 {name} 必须是正整数: {value}")
        return integer_value

    sampling_times = integer_field(HEADER_FRAMES_INDEX, "帧数(header[7])")
    channels_num = integer_field(HEADER_CHANNELS_INDEX, "通道数(header[8])")
    sampling_rate = float(header[HEADER_SAMPLING_RATE_INDEX])
    if not np.isfinite(sampling_rate) or sampling_rate <= 0:
        raise ValueError(f"BIN 文件头采样率(header[9])无效: {sampling_rate}")
    return sampling_times, channels_num, sampling_rate


def read_bin_header(file_path):
    """
    读取 20 位头(80字节)的 .bin 格式 DAS 数据头。

    Returns:
        tuple: (header, sampling_times, channels_num, sampling_rate, endian)
    """
    file_size = os.path.getsize(file_path)

    if file_size <= HEADER_BYTES:
        raise ValueError(f"{file_path}: 文件太短，无法读取。")

    with open(file_path, "rb") as f:
        head_bytes = f.read(HEADER_BYTES)

    # 同时按小端 / 大端解析，根据年月日判断字节序
    hdr_le = np.frombuffer(head_bytes, dtype="<f4", count=HEADER_FLOATS)
    hdr_be = np.frombuffer(head_bytes, dtype=">f4", count=HEADER_FLOATS)

    def looks_like_date(h):
        try:
            y, m, d = int(round(float(h[0]))), int(round(float(h[1]))), int(round(float(h[2])))
            return (2000 <= y <= 2100 and 1 <= m <= 12 and 1 <= d <= 31)
        except:
            return False

    if looks_like_date(hdr_le):
        endian = "<"
        hdr = hdr_le
    elif looks_like_date(hdr_be):
        endian = ">"
        hdr = hdr_be
    else:
        endian = "<"
        hdr = hdr_le

    sampling_times, channels_num, sampling_rate = bin_header_metadata(hdr)

    # 利用文件总长度做交叉验证 (Cross-check)
    body_bytes = file_size - HEADER_BYTES
    if body_bytes % 4 != 0:
        raise ValueError(f"{file_path}: 数据体字节数不是 float32 的整数倍。")
    total_floats = body_bytes // 4

    expected_floats = sampling_times * channels_num
    if expected_floats != total_floats:
        raise ValueError(
            f"{file_path}: 头部帧数={sampling_times}、通道数={channels_num}，"
            f"预期数据量={expected_floats}，实际数据量={total_floats}")

    return hdr, sampling_times, channels_num, sampling_rate, endian


def bin2numpy(file_path, ch1=0, ch2=None):
    """
    直接读取 20 位头(80字节)的 .bin 格式 DAS 数据。

    返回与 bin.ipynb 一致的 (通道数, 采样次数) 形状。
    """
    _header, sampling_times, channels_num, _sampling_rate, endian = read_bin_header(file_path)
    if ch2 is None:
        ch2 = channels_num
    if ch1 < 0 or ch2 > channels_num or ch1 >= ch2:
        raise ValueError(f"{file_path}: 通道范围无效，ch1={ch1}, ch2={ch2}, channels={channels_num}")

    # 一次性读取数据体
    with open(file_path, "rb") as f:
        f.seek(HEADER_BYTES)
        data_bytes = f.read()

    # 按照对应的大小端格式将字节流转为 float32 数组
    dtype_str = endian + "f4"
    data_arr = np.frombuffer(data_bytes, dtype=dtype_str)

    # 原始数据按通道逐帧连续存储，和 notebook 的 reshape 方式保持一致。
    data_matrix = data_arr.reshape((channels_num, sampling_times))

    # 截取你需要的道数范围并返回（加上 astype 确保内存连续性，保护后续 pytorch 运算不报错）
    return data_matrix[ch1:ch2].astype(np.float32, copy=False)



def bin_window(file_path, sample_start, sample_end, ch1=0, ch2=None):
    """Read a channel/time rectangle without materialising the whole BIN.

    The file layout is channel-major, so each requested trace is read as one
    contiguous span.  This keeps long video comparison analysis read-only and
    bounded by its local time window rather than the complete sequence.
    """
    _header, sampling_times, channels_num, _sampling_rate, endian = read_bin_header(file_path)
    start = max(0, int(sample_start))
    end = min(int(sample_end), sampling_times)
    if ch2 is None:
        ch2 = channels_num
    ch1, ch2 = int(ch1), int(ch2)
    if not (0 <= ch1 < ch2 <= channels_num) or end <= start:
        raise ValueError(f"{file_path}: 窗口或通道范围无效")
    count = end - start
    result = np.empty((ch2 - ch1, count), dtype=np.float32)
    item_size = np.dtype(endian + "f4").itemsize
    with open(file_path, "rb") as stream:
        for output_channel, source_channel in enumerate(range(ch1, ch2)):
            offset = HEADER_BYTES + (source_channel * sampling_times + start) * item_size
            stream.seek(offset)
            values = np.fromfile(stream, dtype=endian + "f4", count=count)
            if values.size != count:
                raise ValueError(f"{file_path}: BIN 窗口数据不足")
            result[output_channel] = values
    return result


In [ ]:
%%copied_module _das_veh_lab_snapshot.utils.classes.data_group
"""Data-group metadata and memory guards for stitched DAS files."""

from __future__ import annotations

import ctypes
import os
import re
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable, List, Optional, Sequence, Tuple


def natural_sort_key(value: str) -> Tuple[Tuple[int, object], ...]:
    """Return a case-insensitive key that keeps numeric filename parts ordered."""

    text = str(value)
    return tuple(
        (1, int(part)) if part.isdigit() else (0, part.casefold())
        for part in re.split(r"(\d+)", text)
        if part
    )


@dataclass(frozen=True)
class FileSegment:
    """The sample interval occupied by one source file in a stitched array."""

    path: str
    start_sample: int
    sample_count: int

    def __post_init__(self) -> None:
        if int(self.start_sample) < 0:
            raise ValueError("文件分段起始采样点不能小于 0")
        if int(self.sample_count) <= 0:
            raise ValueError("文件分段采样点数必须大于 0")

    @property
    def end_sample(self) -> int:
        """Exclusive global sample end."""

        return int(self.start_sample) + int(self.sample_count)

    @property
    def name(self) -> str:
        return Path(self.path).name

    @property
    def sample_range(self) -> Tuple[int, int]:
        return int(self.start_sample), self.end_sample


@dataclass
class DataGroup:
    """Description of the source files represented by one viewer data array."""

    channel_count: int
    sampling_rate: float
    segments: List[FileSegment] = field(default_factory=list)

    def __post_init__(self) -> None:
        if int(self.channel_count) <= 0:
            raise ValueError("数据组通道数必须大于 0")
        if float(self.sampling_rate) <= 0:
            raise ValueError("数据组采样率必须大于 0")
        expected_start = 0
        for segment in self.segments:
            if segment.start_sample != expected_start:
                raise ValueError("数据组文件分段必须连续且不能重叠")
            expected_start = segment.end_sample

    @classmethod
    def from_files(
        cls,
        file_paths: Sequence[str],
        sample_counts: Sequence[int],
        channel_count: int,
        sampling_rate: float,
    ) -> "DataGroup":
        if len(file_paths) != len(sample_counts) or not file_paths:
            raise ValueError("文件路径和采样点数必须一一对应且不能为空")
        start = 0
        segments: List[FileSegment] = []
        for path, count in zip(file_paths, sample_counts):
            segment = FileSegment(str(path), start, int(count))
            segments.append(segment)
            start = segment.end_sample
        return cls(int(channel_count), float(sampling_rate), segments)

    @property
    def total_samples(self) -> int:
        return self.segments[-1].end_sample if self.segments else 0

    @property
    def segment_ranges(self) -> List[Tuple[int, int]]:
        return [segment.sample_range for segment in self.segments]

    @property
    def boundaries(self) -> List[int]:
        return [segment.start_sample for segment in self.segments[1:]]


def available_memory_bytes() -> Optional[int]:
    """Return currently available physical memory when the platform exposes it."""

    if sys.platform == "win32":
        class MemoryStatusEx(ctypes.Structure):
            _fields_ = [
                ("dwLength", ctypes.c_ulong),
                ("dwMemoryLoad", ctypes.c_ulong),
                ("ullTotalPhys", ctypes.c_ulonglong),
                ("ullAvailPhys", ctypes.c_ulonglong),
                ("ullTotalPageFile", ctypes.c_ulonglong),
                ("ullAvailPageFile", ctypes.c_ulonglong),
                ("ullTotalVirtual", ctypes.c_ulonglong),
                ("ullAvailVirtual", ctypes.c_ulonglong),
                ("ullAvailExtendedVirtual", ctypes.c_ulonglong),
            ]

        status = MemoryStatusEx()
        status.dwLength = ctypes.sizeof(status)
        try:
            if ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(status)):
                return int(status.ullAvailPhys)
        except (AttributeError, OSError):
            return None

    try:
        page_size = int(os.sysconf("SC_PAGE_SIZE"))
        available_pages = int(os.sysconf("SC_AVPHYS_PAGES"))
    except (AttributeError, OSError, ValueError):
        return None
    return page_size * available_pages


def estimate_array_bytes(
    channel_count: int,
    sample_count: int,
    copies: float = 1.0,
) -> int:
    """Estimate float32 array memory for a workflow with multiple live copies."""

    return int(int(channel_count) * int(sample_count) * 4 * float(copies))


def ensure_memory_budget(
    channel_count: int,
    sample_count: int,
    copies: float,
    operation: str,
    maximum_fraction: float = 0.75,
) -> Tuple[int, Optional[int]]:
    """Reject work that is likely to exhaust currently available memory."""

    required = estimate_array_bytes(channel_count, sample_count, copies)
    available = available_memory_bytes()
    if available is not None and required > int(available * maximum_fraction):
        raise MemoryError(
            f"{operation}预计至少需要 {format_bytes(required)} 内存，"
            f"当前可用约 {format_bytes(available)}；请减少文件数、通道或时间范围。"
        )
    return required, available


def format_bytes(value: int) -> str:
    size = float(value)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if size < 1024.0 or unit == "TiB":
            return f"{size:.1f} {unit}"
        size /= 1024.0
    return f"{size:.1f} TiB"


**A2. 跨文件读取与结果类型：按完整定义原样复制。** 包括跨文件窗口、取消检查、候选结果和输出折线简化。

In [ ]:
%%copied_module _das_veh_lab_snapshot.utils.classes.video_trajectory
from __future__ import annotations


from typing import Dict, List, Tuple


import numpy as np


from .data_group import DataGroup, ensure_memory_budget


from ..bin_reader import bin_window


def read_group_window(
    data_group: DataGroup,
    start_sample: int,
    end_sample: int,
    channel_from: int,
    channel_to: int,
) -> np.ndarray:
    """Read one global window across BIN boundaries without loading full BINs."""

    start = max(0, int(start_sample))
    end = min(int(end_sample), data_group.total_samples)
    channel_from = max(1, int(channel_from))
    channel_to = min(int(channel_to), data_group.channel_count)
    if end <= start or channel_from > channel_to:
        raise ValueError("连续 DAS 窗口或通道范围无效")
    chunks: List[np.ndarray] = []
    for segment in data_group.segments:
        local_start = max(start, segment.start_sample)
        local_end = min(end, segment.end_sample)
        if local_end <= local_start:
            continue
        chunks.append(
            bin_window(
                segment.path,
                local_start - segment.start_sample,
                local_end - segment.start_sample,
                channel_from - 1,
                channel_to,
            )
        )
    if not chunks:
        raise ValueError("没有与连续 DAS 窗口重叠的 BIN 文件")
    result = np.concatenate(chunks, axis=1)
    expected = end - start
    if result.shape != (channel_to - channel_from + 1, expected):
        raise ValueError("跨文件窗口读取不连续；不会把不完整数据当作无车")
    return result


In [ ]:
%%copied_module _das_veh_lab_snapshot.utils.classes.trajectory_prescreen
from dataclasses import dataclass


import numpy as np


class PrescreenCancelled(Exception):
    pass


def check_cancel(cancel):
    if cancel is not None and cancel.is_set():
        raise PrescreenCancelled()


@dataclass
class TrackProposal:
    identifier: int
    vertices: list
    continuity: float
    strength: float
    ambiguous: bool = False
    edited: bool = False


@dataclass(frozen=True)
class PrescreenResult:
    proposals: tuple
    notes: tuple = ()


def simplify_track(points, sampling_rate, tolerance_seconds=0.08, tolerance_channels=0.75):
    """RDP simplification in scaled time/channel units; preserve endpoints."""
    points = np.asarray(points, dtype=float)
    if len(points) <= 2:
        return [[int(round(p[0])), float(p[1])] for p in points]
    scaled = points / [sampling_rate * tolerance_seconds, tolerance_channels]
    keep = {0, len(points) - 1}
    pending = [(0, len(points) - 1)]
    while pending:
        first, last = pending.pop()
        if last - first < 2:
            continue
        a, b = scaled[first], scaled[last]
        delta = b - a
        inner = scaled[first + 1:last]
        ratio = np.clip((inner - a) @ delta / max(float(delta @ delta), 1e-12), 0, 1)
        distance = np.linalg.norm(inner - a - ratio[:, None] * delta, axis=1)
        index = int(np.argmax(distance))
        if distance[index] > 1:
            middle = first + 1 + index
            keep.add(middle)
            pending.extend(((first, middle), (middle, last)))
    return [[int(round(points[i, 0])), float(points[i, 1])] for i in sorted(keep)]


**A3. 上游工具和核心跟踪类：整文件复制。** 当前软件对上游代码的适配已经包含在这里。

In [ ]:
%%copied_module _das_veh_lab_snapshot.utils.vendor.das_veh.car_tracking_utils
import os
import numpy as np
from scipy.stats import norm
from scipy.signal import fftconvolve
import matplotlib.pyplot as plt


def read_data(data_dir, data_name):
    data_path = os.path.join(data_dir, data_name)
    data_file = np.load(data_path)
    return data_file["data"], data_file["x_axis"], data_file["t_axis"]

def read_from_csv_name(data_dir, data_name):
    data_dir_w_name = os.path.join(data_dir, data_name)
    das_channel = np.genfromtxt(data_dir_w_name + '_x_axis.csv')
    das_time = np.genfromtxt(data_dir_w_name + '_t_axis.csv')
    data = np.genfromtxt(data_dir_w_name + '.csv', delimiter=' ')
    return data, das_channel, das_time


def likelihood_1d(peak_loc, das_time_ds, sigma):
    data_tmp_thrd = np.zeros(len(das_time_ds))
    if len(das_time_ds) > 100000 and len(peak_loc):
        # Evaluate the same Gaussian likelihood on every original sample.
        # FFT convolution avoids one full array per peak; no resampling occurs.
        dt = float(das_time_ds[1] - das_time_ds[0])
        radius = int(np.ceil(8 * sigma / dt))
        kernel = norm.pdf(np.arange(-radius, radius + 1) * dt, scale=sigma)
        np.add.at(data_tmp_thrd, peak_loc, 1)
        result = fftconvolve(data_tmp_thrd, kernel, mode="same")
        # Upstream accepts any positive likelihood. Numerical FFT round-off
        # must not manufacture seed peaks where the Gaussian sum is zero.
        result[result < max(float(result.max()), 0) * 1e-12] = 0
        return result
    for j in range(len(peak_loc)):
        data_tmp_thrd = data_tmp_thrd + norm.pdf(das_time_ds, loc=das_time_ds[peak_loc[j]], scale=sigma)

    return data_tmp_thrd

def interp_nan_value(veh_states):
    for k, state in enumerate(veh_states):
        # Find indices of non-NaN values
        non_nan_indices = np.where(~np.isnan(state))[0]
        # Generate array of indices
        indices = np.arange(len(state))
        if not len(non_nan_indices):
            continue
        # Replace NaN values with linearly interpolated values
        state[np.isnan(state)] = np.interp(np.isnan(state).nonzero()[0], non_nan_indices, state[non_nan_indices])


def remove_unrealistic_tracking(veh_base, veh_states, adjacency_nan_count_lim=20, factor=1):
    invalid_num_tmp = []
    veh_states = veh_states[:, ::factor]
    for v in range(len(veh_base)):
        tmp = veh_states[v][~np.isnan(veh_states[v])]

        nan_indices = np.where(np.isnan(veh_states[v]))[0]
        diffs = np.diff(nan_indices)
        adjacency_count = np.sum(diffs == 1)

        # remove unrealistic cases
        if len(~np.isnan(tmp)) < 0.3 * len(veh_states[v]) or sum(
               np.convolve(np.diff(tmp), np.ones(20), mode='valid') <= -15) or abs(sum(np.diff(tmp))) < 30 * (
               len(tmp) / len(veh_states[v])) or adjacency_count >= adjacency_nan_count_lim:

            invalid_num_tmp.append(v)

        tmp_idx = np.where(~np.isnan(veh_states[v]))[0]
        invalid_idx = np.where(abs(np.diff(tmp)) > 20)[0]
        veh_states[v][tmp_idx[invalid_idx + 1]] = np.nan

    valid_num_tmp = list(range(len(veh_base)))
    for v in invalid_num_tmp:
        valid_num_tmp.remove(v)
    tracked_v = veh_states[valid_num_tmp, :]
    return tracked_v

def plot_data(data, x_axis, t_axis, pclip=98, ax=None, figsize=(10, 10), y_lim=None, x_lim=None):
    vmax = np.percentile(np.abs(data), pclip)
    if not ax:
        fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(data.T,
              aspect="auto",
              extent=[x_axis[0], x_axis[-1], t_axis[-1], t_axis[0]],
              cmap="gray",
              vmax=vmax,
              vmin=-vmax)
    ax.set_ylim(y_lim)
    ax.set_xlim(x_lim)





In [ ]:
%%copied_module _das_veh_lab_snapshot.utils.vendor.das_veh.tracking
"""
Kalman Filter code is based on https://github.com/jingxiaoliu/TelecomTM
"""

import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import numpy as np
import os
from .car_tracking_utils import likelihood_1d, remove_unrealistic_tracking, plot_data, interp_nan_value
from collections.abc import Iterable
import time


class KF_tracking:

    def __init__(self, data, t_axis, x_axis, args):
        self.data = data
        self.t_axis = t_axis
        self.x_axis = x_axis
        self.dx = x_axis[1] - x_axis[0]
        self.args = args
        self.check_cancel = args.get("check_cancel", lambda: None)
        self.progress = args.get("progress", lambda fraction: None)

    def detect_in_one_section(self, start_x, nx=15, detection_args=None, sigma=0.1, pclip=98, show_plot=False, plt_xlim=1000):
        if not detection_args:
            detection_args = self.args["detect"]

        minprominence = detection_args["minprominence"]
        minseparation = detection_args["minseparation"]
        prominenceWindow = detection_args["prominenceWindow"]

        peak_erode = np.zeros(len(self.t_axis))

        start_x_idx = np.argmin(np.abs(start_x - self.x_axis))
        # a quick detection using the first 15 channels
        for i in range(min(nx, len(self.x_axis) - start_x_idx)):
            self.check_cancel()
            das_for_dt = self.data[start_x_idx+i]
            cached = self.args.get("peak_locations")
            peak_loc = cached[start_x_idx+i] if cached is not None else find_peaks(das_for_dt,
                                  prominence=minprominence,
                                  wlen=prominenceWindow,
                                  distance=minseparation)[0]
                                  # height=height)[0]
            peak_erode_tmp = likelihood_1d(peak_loc, self.t_axis, sigma)
            peak_erode += peak_erode_tmp

        peak_loc_tmp, _ = find_peaks(peak_erode, height=max(peak_erode) * 0., distance=minseparation)

        if show_plot:
            vmax = np.percentile(np.abs(self.data), pclip)
            fig, axes = plt.subplots(1, 2, figsize=(15, 10), gridspec_kw={'width_ratios':[3,1]}, sharey=True)
            axes[0].imshow(self.data.T,
                           aspect="auto",
                           extent=[self.x_axis[0], self.x_axis[-1], self.t_axis[-1], self.t_axis[0]],
                           cmap="gray",
                           vmax=vmax,
                           vmin=-vmax)
            axes[0].axvline(x=self.x_axis[start_x_idx], c='r')
            axes[0].axvline(x=self.x_axis[start_x_idx+nx], c='b')
            axes[1].plot(peak_erode, self.t_axis, 'b')
            axes[1].plot(peak_erode[peak_loc_tmp], self.t_axis[peak_loc_tmp], 'r^')
            axes[0].plot([self.x_axis[start_x_idx+nx//2]] * len(peak_loc_tmp), self.t_axis[peak_loc_tmp], 'r^')
            axes[0].set_xlim([self.x_axis[0], plt_xlim])

        veh_base = peak_loc_tmp
        return veh_base

    def tracking_with_veh_base(self, start_x, end_x, veh_base, sigma_a=0.01, detection_args=None):
        if detection_args is None:
            detection_args = self.args['detect']

        start_x_idx = np.argmin(np.abs(start_x - self.x_axis))
        end_x_idx = np.argmin(np.abs(end_x - self.x_axis))

        x_axis = self.x_axis[start_x_idx: end_x_idx + 1]

        minprominence = detection_args["minprominence"]
        minseparation = detection_args["minseparation"]
        prominenceWindow = detection_args["prominenceWindow"]
        height = detection_args.get("height", None)

        veh_states = np.empty((len(veh_base), end_x_idx - start_x_idx + 1))
        veh_states[:] = np.nan
        veh_states_v = np.empty((len(veh_base), end_x_idx - start_x_idx + 1))
        veh_states_v[:] = np.nan

        R = self.args.get("measurement_variance", 1)
        Tk1k = np.empty((2, len(veh_base)))
        Tk1k[:] = np.nan
        Tkk = np.empty((2, len(veh_base)))
        Tkk[:] = np.nan
        Pkk = np.empty((2, 2, len(veh_base)))
        Pkk[:] = np.nan
        Pk1k = np.empty((2, 2, len(veh_base)))
        Pk1k[:] = np.nan
        Xv = np.empty(len(veh_base))
        Xv[:] = np.nan
        C = np.array([1, 0])
        veh_base_state = veh_base.astype(float, copy=True)
        missed = np.zeros(len(veh_base), dtype=int)
        active = np.ones(len(veh_base), dtype=bool)
        st = time.time()

        factor = self.args.get("channel_step", 3)

        for i in range(start_x_idx, end_x_idx + 1, factor):
            self.check_cancel()
            self.progress((i - start_x_idx + 1) / len(x_axis))
            for v in range(len(veh_base)):
                if not active[v]:
                    continue

                if sum(~np.isnan(veh_states[v, :])) == 1:
                    Tkk[:, v] = [float(veh_states[v, ~np.isnan(veh_states[v, :])][0]), 0]
                    # raise Exception
                    Xv[v] = float(x_axis[~np.isnan(veh_states[v, :])][0])
                    Pkk[:, :, v] = np.array([[0, 0], [0, 0]])
                    veh_base_state[v] = veh_base[v]
                elif sum(~np.isnan(veh_states[v, :])) == 0:
                    veh_base_state[v] = veh_base[v]
                else:
                    if self.args.get("initialize_slope", False) and sum(~np.isnan(veh_states[v, :])) == 2:
                        indices = np.flatnonzero(~np.isnan(veh_states[v, :]))
                        last = indices[-1]
                        Tkk[:, v] = [veh_states[v, last],
                            (veh_states[v, last] - veh_states[v, indices[0]]) / (x_axis[last] - x_axis[indices[0]])]
                        Xv[v] = x_axis[last]
                        Pkk[:, :, v] = np.diag([R, R / self.dx ** 2])
                    delta_x = self.x_axis[i] - Xv[v]
                    A = [[1, delta_x], [0, 1]]
                    Q = sigma_a * np.array([[0.25 * delta_x ** 4, 0.5 * delta_x ** 3], [0.5 * delta_x ** 3, delta_x ** 2]])
                    Tk1k[:, v] = np.matmul(A, Tkk[:, v])

                    Pk1k[:, :, v] = np.matmul(np.matmul(A, Pkk[:, :, v]), np.transpose(A)) + Q
                    veh_base_state[v] = Tk1k[0, v]

            das_for_dt = self.data[i]
            cached = self.args.get("peak_locations")
            peak_loc = cached[i] if cached is not None else find_peaks(
                das_for_dt, prominence=minprominence, wlen=prominenceWindow, distance=minseparation)[0]

            for p in range(len(veh_base_state)):
                if not active[p]:
                    continue
                dist_tmp = peak_loc - veh_base_state[p]
                left, right = self.args.get("association_window", (-15, 30))
                idx_tmp = np.where((dist_tmp > left) & (dist_tmp <= right))[0]
                valid_tmp = dist_tmp[idx_tmp]
                if len(valid_tmp):
                    # Preserve upstream positive-first association unless the
                    # adapter explicitly requests nearest-residual matching.
                    positive = np.flatnonzero(valid_tmp > 0)
                    choices = positive if len(positive) and not self.args.get("nearest_peak", False) else np.arange(len(valid_tmp))
                    chosen = choices[np.argmin(np.abs(valid_tmp[choices]))]
                    veh_states[p, i - start_x_idx] = float(peak_loc[idx_tmp[chosen]])
                    missed[p] = 0
                else:
                    missed[p] += 1
                    limit = self.args.get("maximum_missed_channels")
                    if limit is not None and missed[p] > limit:
                        active[p] = False

            # filtering
            for v in range(len(veh_base)):

                if active[v] and (sum(~np.isnan(veh_states[v, :])) > 2) and (not np.isnan(veh_states[v, i - start_x_idx])):
                    K = Pk1k[:, :, v] @ C.T / (R + C @ Pk1k[:, :, v] @ C.T)

                    # if i + 1 == 266 and v + 1 == 1:
                    #     print(Pk1k[:, :, v])
                    tkk = Tk1k[:, v] + K * (veh_states[v, i - start_x_idx] - C @ Tk1k[:, v])
                    # print("tkk",tkk.shape)
                    Tkk[:, v] = tkk
                    Pkk[:, :, v] = Pk1k[:, :, v] - (K.reshape(2, 1) @ C.reshape(1, 2)) @ Pk1k[:, :, v]
                    Xv[v] = self.x_axis[i]


            # if i + 1 == 280:
            #     print(tkk)
        if self.args.get("observed_only", False):
            return veh_states
        tracked_v = remove_unrealistic_tracking(veh_base, veh_states, factor=factor)
        # tracked_v = veh_states
        tracked_v_full = np.empty((tracked_v.shape[0], len(x_axis)))
        tracked_v_full[:] = np.nan
        tracked_v_full[:, ::factor] = tracked_v[:, :tracked_v_full[:, ::factor].shape[1]]
        # tracked_v = veh_states
        interp_nan_value(tracked_v_full)

        return tracked_v_full

    def tracking_visulization_one_section(self, start_x, tracked_v, plt_xlim=800, plt_tlim=78, t_min=0, ax=None,
                                          plot_tracking=True, plt_xlo=0, fontsize=16, tickfont=12, fig_dir=None, fig_name=None):
        start_x_idx = np.argmin(np.abs(start_x - self.x_axis))
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 10))
        self.plot_data(ax=ax)

        if plot_tracking:
            for v in range(len(tracked_v[:, 1])):
                tmp = tracked_v[v][~np.isnan(tracked_v[v, :])].astype(int)
                dist_idx_tmp = np.where(~np.isnan(tracked_v[v, :]))[0] + start_x_idx
                ax.plot(self.x_axis[dist_idx_tmp], self.t_axis[tmp], '.', color='red', markersize=1)

        ax.set_xlim([plt_xlo, plt_xlim])
        ax.set_ylim([plt_tlim, t_min])
        ax.set_xlabel("Distance along fiber [m]", fontsize=fontsize)
        ax.set_ylabel("Time [s]", fontsize=fontsize)
        ax.tick_params(axis='both', which='major', labelsize=tickfont)
        if fig_dir and fig_name:
            fig_path = os.path.join(fig_dir, fig_name)
            plt.savefig(fig_path)
            print(f'{fig_path} is saved...')

    def plot_data(self, pclip=98, ax=None):
        x_axis, t_axis = self.x_axis, self.t_axis
        plot_data(self.data, x_axis, t_axis, pclip=pclip, ax=ax)

    def show_detection_example(self, x0=720, detection_args=None, pclip=98):
        if not detection_args:
            detection_args = self.args["detect"]

        if not isinstance(x0, Iterable):
            x0 = [x0]

        markers = ['r^', 'bv']

        minprominence = detection_args["minprominence"]
        minseparation = detection_args["minseparation"]
        prominenceWindow = detection_args["prominenceWindow"]
        height = detection_args.get("height", None)

        fig, axes = plt.subplots(1, 1+len(x0), gridspec_kw={'width_ratios': [3] + [1] * len(x0)}, sharey=True, figsize=(5*(2+len(x0)), 10))
        vmax = np.percentile(np.abs(self.data), pclip)

        axes[0].imshow(self.data.T,
                       aspect="auto",
                       extent=[self.x_axis[0], self.x_axis[-1], self.t_axis[-1], self.t_axis[0]],
                       cmap="gray",
                       vmax=vmax,
                       vmin=-vmax)

        for k, (x, m) in enumerate(zip(x0, markers)):
            x_idx = np.argmin(np.abs(x - self.x_axis))
            das_for_dt = self.data[x_idx]
            peak_loc, _ = find_peaks(das_for_dt, prominence=minprominence,
                                     distance=minseparation, wlen=prominenceWindow, height=height)

            axes[0].plot([self.x_axis[x_idx]]*len(peak_loc), self.t_axis[peak_loc], m)

            axes[k+1].plot(das_for_dt, self.t_axis, 'k')
            axes[k+1].plot(das_for_dt[peak_loc], self.t_axis[peak_loc], m)
            axes[k+1].axvline(x=minprominence, linestyle='--')
            plt.xlabel('Time (s)')
            plt.ylabel('Telecom fiber response')
            plt.legend(['Response', 'Vehicle arrival'], loc='lower right')

        axes[0].set_xlim([self.x_axis[0], 1000])
        plt.show()








**A4. 软件的全采样率适配器：整文件复制。** 默认参数、滤波、归一化、起点预算、关联、速度筛选、去重都由它执行。

In [ ]:
%%copied_module _das_veh_lab_snapshot.utils.classes.das_veh_tracking
"""Full-sampling-rate adapter for vendored das_veh KF_tracking.

No decimation, temporal resampling, spatial interpolation, or annotation input
is used. Each peak index addresses an acquired sample in an original channel.
"""

from dataclasses import replace

import numpy as np
from scipy import signal

from ..vendor.das_veh.tracking import KF_tracking
from .data_group import ensure_memory_budget
from .video_trajectory import read_group_window
from .trajectory_prescreen import (
    PrescreenResult, TrackProposal, check_cancel, simplify_track,
)


DEFAULT_DAS_VEH_PARAMETERS = {
    'algorithm': 'das_veh', 'channel_from': 290, 'channel_to': 520,
    'frequency_low': 0.01, 'frequency_high': 1.0, 'channel_spacing': 4.0,
    'minimum_speed': 0.0, 'maximum_speed': 120 / 3.6,
    'peak_prominence': 1.5, 'minimum_separation': 1.0,
    'prominence_window': 12.0, 'seed_sigma': 0.08, 'seed_width': 3,
    'seed_stride': 32, 'maximum_seeds': 1000, 'maximum_candidates': 100,
    'minimum_track_channels': 20, 'maximum_missed_channels': 5,
    'tracking_tolerance': 1.2, 'sigma_a': 0.01,
    'direction': 'auto', 'polarity': 'both',
}


def _options(parameters):
    options = {**DEFAULT_DAS_VEH_PARAMETERS, **parameters}
    for name in ('frequency_low', 'frequency_high', 'channel_spacing',
                 'minimum_speed', 'maximum_speed', 'peak_prominence',
                 'minimum_separation', 'prominence_window', 'seed_sigma',
                 'tracking_tolerance', 'sigma_a'):
        options[name] = float(options[name])
        if not np.isfinite(options[name]):
            raise ValueError(f'参数 {name} 必须是有限数值')
    if options['minimum_speed'] < 0 or options['maximum_speed'] <= options['minimum_speed']:
        raise ValueError('投影速度需满足 0 ≤ 下限 < 上限；下限 0 表示不限制最低速度')
    for name in ('channel_spacing', 'peak_prominence', 'minimum_separation',
                 'prominence_window', 'seed_sigma', 'tracking_tolerance', 'sigma_a'):
        if options[name] <= 0:
            raise ValueError(f'参数 {name} 必须大于 0')
    for name in ('seed_width', 'seed_stride', 'maximum_seeds', 'maximum_candidates',
                 'minimum_track_channels', 'maximum_missed_channels'):
        value = float(options[name])
        if not np.isfinite(value) or value != int(value) or value < (0 if name == 'maximum_missed_channels' else 1):
            raise ValueError(f'参数 {name} 的整数范围无效')
        options[name] = int(value)
    if options['minimum_track_channels'] < 3:
        raise ValueError('最短轨迹至少需要 3 个有效通道')
    if options['direction'] not in ('auto', 'increasing', 'decreasing'):
        raise ValueError('搜索方向无效')
    if options['polarity'] not in ('both', 'positive', 'negative'):
        raise ValueError('信号极性无效')
    return options


def _is_duplicate(proposal, existing, fs, tolerance=0.15):
    """Require substantial shared extent; keep extensions and brief crossings."""
    a = np.asarray(proposal.vertices, dtype=float)
    for other in existing:
        b = np.asarray(other.vertices, dtype=float)
        if (a[-1, 0] - a[0, 0]) * (b[-1, 0] - b[0, 0]) <= 0:
            continue
        first, last = max(a[0, 1], b[0, 1]), min(a[-1, 1], b[-1, 1])
        if last <= first or (last - first) < 0.7 * (a[-1, 1] - a[0, 1]):
            continue
        channels = np.arange(np.ceil(first), np.floor(last) + 1)
        differences = np.abs(np.interp(channels, a[:, 1], a[:, 0]) -
                             np.interp(channels, b[:, 1], b[:, 0])) / fs
        if len(differences) and np.mean(differences <= tolerance) >= 0.8:
            return True
    return False


def detect_das_veh_tracks(data, sampling_rate, parameters=None, *, sample_origin=0,
                          channel_origin=1, sample_bounds=None, cancel=None, progress=None):
    """Run upstream detection/association on normalized full-rate traces."""
    options = _options(parameters or {})
    fs = float(sampling_rate)
    if not np.isfinite(fs) or fs <= 0:
        raise ValueError('采样率无效')
    array = np.asarray(data)
    if array.ndim != 2 or min(array.shape) < 2 or not np.isfinite(array).all():
        raise ValueError('跟踪需要有限数值的二维 DAS 数据')
    check_cancel(cancel)
    progress = progress or (lambda percent, message: None)
    detection = {
        'minprominence': options['peak_prominence'],
        'minseparation': max(1, int(round(options['minimum_separation'] * fs))),
        'prominenceWindow': max(3, int(round(options['prominence_window'] * fs)) | 1),
    }
    signs = (1, -1) if options['polarity'] == 'both' else ((1,) if options['polarity'] == 'positive' else (-1,))
    directions = (1, -1) if options['direction'] == 'auto' else ((1,) if options['direction'] == 'increasing' else (-1,))
    n_channels = array.shape[0]
    t_axis = np.arange(array.shape[1], dtype=float) / fs
    x_axis = np.arange(n_channels, dtype=float) * options['channel_spacing']
    proposals = []
    seed_budget = options['maximum_seeds']
    used_seeds = 0
    budget_hit = False
    pass_count = len(signs) * len(directions)
    for sign_index, sign in enumerate(signs):
        peaks, strengths = [], []
        for row, trace in enumerate(array):
            check_cancel(cancel)
            locations, properties = signal.find_peaks(
                trace if sign == 1 else -trace,
                prominence=detection['minprominence'],
                distance=detection['minseparation'], wlen=detection['prominenceWindow'])
            peaks.append(locations)
            strengths.append(properties['prominences'])
            progress(35 + int(64 * (sign_index + 0.15 * (row + 1) / n_channels) / len(signs)), '在原始采样点检测峰值')
        for direction_index, direction in enumerate(directions):
            check_cancel(cancel)
            oriented_peaks = peaks if direction > 0 else peaks[::-1]
            oriented_strengths = strengths if direction > 0 else strengths[::-1]
            # A reversed view changes traversal only, never the time grid or dx.
            oriented_data = array if direction > 0 else array[::-1]
            arguments = {
                'detect': detection, 'peak_locations': oriented_peaks,
                'channel_step': 1, 'measurement_variance': (fs / 50) ** 2,
                'association_window': (-options['tracking_tolerance'] * fs,
                                       options['tracking_tolerance'] * fs),
                'initialize_slope': True, 'nearest_peak': True, 'observed_only': True,
                'maximum_missed_channels': options['maximum_missed_channels'],
                'check_cancel': lambda: check_cancel(cancel),
            }
            tracker = KF_tracking(oriented_data, t_axis, x_axis, arguments)
            starts = list(range(0, n_channels - options['minimum_track_channels'] + 1, options['seed_stride']))
            job_limit = max(1, seed_budget // max(1, pass_count * len(starts)))
            for start_index, start in enumerate(starts):
                check_cancel(cancel)
                def report(fraction):
                    progress(35 + int(64 * (sign_index + 0.15 + 0.85 *
                             (direction_index + (start_index + fraction) / max(1, len(starts))) / len(directions)) / len(signs)),
                             '原版卡尔曼逐通道跟踪（原始采样率）')
                arguments['progress'] = report
                tracker.progress = report
                bases = tracker.detect_in_one_section(x_axis[start], nx=options['seed_width'],
                                                       sigma=options['seed_sigma'], show_plot=False)
                if sample_bounds is not None:
                    samples = sample_origin + bases
                    bases = bases[(samples >= sample_bounds[0]) & (samples < sample_bounds[1])]
                remaining = min(job_limit, max(0, seed_budget - used_seeds))
                if len(bases) > remaining:
                    budget_hit = True
                    # Spread budget over space, direction, polarity, and time.
                    bases = bases[np.linspace(0, len(bases) - 1, remaining).astype(int)]
                used_seeds += len(bases)
                if not len(bases):
                    continue
                states = tracker.tracking_with_veh_base(x_axis[start], x_axis[-1], bases,
                                                        sigma_a=options['sigma_a'] * (fs / 50) ** 2)
                for state in states:
                    check_cancel(cancel)
                    rows = np.flatnonzero(np.isfinite(state)) + start
                    times = state[np.isfinite(state)]
                    global_samples = np.rint(sample_origin + times).astype(np.int64)
                    keep = np.ones(len(rows), dtype=bool)
                    if sample_bounds is not None:
                        keep = (global_samples >= sample_bounds[0]) & (global_samples < sample_bounds[1])
                    rows, times, global_samples = rows[keep], times[keep], global_samples[keep]
                    if len(rows) < options['minimum_track_channels']:
                        continue
                    centered = rows - rows.mean()
                    slope = float(centered @ (times - times.mean()) / (centered @ centered)) / fs
                    if slope <= 0:
                        continue
                    speed = options['channel_spacing'] / slope
                    if speed > options['maximum_speed'] or speed < options['minimum_speed']:
                        continue
                    # Filter local sustained slope reversals / extreme speed,
                    # allowing individual waveform peak jitter and crossings.
                    if len(rows) >= 9:
                        slopes = (times[8:] - times[:-8]) / (rows[8:] - rows[:-8]) / fs
                        minimum_slope = options['channel_spacing'] / options['maximum_speed']
                        if np.mean(slopes < minimum_slope * 0.5) > 0.2:
                            continue
                    global_channels = (rows if direction > 0 else n_channels - 1 - rows) + channel_origin
                    order = np.argsort(global_channels)
                    points = np.column_stack((global_samples[order], global_channels[order]))
                    response = []
                    for row, time in zip(rows, times):
                        index = np.searchsorted(oriented_peaks[row], int(round(time)))
                        if index < len(oriented_strengths[row]):
                            response.append(float(oriented_strengths[row][index]))
                    continuity = len(rows) / (rows[-1] - rows[0] + 1)
                    jitter = np.std(times / fs - (rows - rows[0]) * slope)
                    proposals.append(TrackProposal(0, points.tolist(), continuity,
                                                  float(np.mean(response)) if response else 0,
                                                  bool(jitter > options['tracking_tolerance'])))
    proposals.sort(key=lambda p: (-(p.vertices[-1][1] - p.vertices[0][1]), -p.strength))
    retained = []
    for proposal in proposals:
        check_cancel(cancel)
        if not _is_duplicate(proposal, retained, fs):
            retained.append(proposal)
    notes = [f'das_veh 原版核心；{fs:g} Hz 全采样点处理；正负极性分别检测，不使用人工标注约束']
    if budget_hit:
        notes.append('达到起点预算，部分起点未搜索；可缩小时间范围后重试')
    if len(retained) > options['maximum_candidates']:
        notes.append(f'候选超过上限，仅保留最长的 {options["maximum_candidates"]} 条')
    if not retained:
        notes.append('未检出满足条件的轨迹，不代表范围内无车；可调整灵敏度、最短轨迹或容差后重试')
    retained = retained[:options['maximum_candidates']]
    retained.sort(key=lambda p: min(v[0] for v in p.vertices))
    result = tuple(replace(p, identifier=i, vertices=simplify_track(p.vertices, fs))
                   for i, p in enumerate(retained, 1))
    return PrescreenResult(result, tuple(notes))


def analyze_das_veh(group, raw_data, visible_start, visible_end, channel_from, channel_to,
                    parameters, cancel=None, progress=None):
    """Read padded blocks and filter every original sample at the supplied fs."""
    options = _options(parameters)
    fs = float(group.sampling_rate)
    if not np.isfinite(fs) or fs <= 0:
        raise ValueError('采样率无效')
    if channel_from < 1 or channel_to > group.channel_count or channel_to - channel_from + 1 < 8:
        raise ValueError('检测范围至少需要 8 个有效通道，且不能超出来源通道范围')
    if not 0 <= visible_start < visible_end <= group.total_samples:
        raise ValueError('检测时间范围无效')
    low, high = options['frequency_low'], options['frequency_high']
    if not 0 < low < high < fs / 2:
        raise ValueError('滤波频率需满足 0 < 低频 < 高频 < 原始采样率的一半')
    if raw_data is not None and raw_data.shape != (group.channel_count, group.total_samples):
        raise ValueError('原始数据与 DAS 来源范围不一致，请重新载入数据')
    check_cancel(cancel)
    progress = progress or (lambda percent, message: None)
    padding = int(np.ceil(fs / low))
    start, end = max(0, int(visible_start) - padding), min(group.total_samples, int(visible_end) + padding)
    if (end - start) / fs < 2 / low:
        raise ValueError(f'当前来源不足 {2 / low:g} 秒，无法稳定使用 {low:g} Hz；不会自动改变滤波频段')
    notes = []
    if start > visible_start - padding or end < visible_end + padding:
        notes.append('来源边界限制了滤波补充数据，请复查窗口两端候选')
    rows, samples = channel_to - channel_from + 1, end - start
    ensure_memory_budget(rows, samples, 6.0, '全采样率车辆跟踪')
    # SOS preserves numerical stability for 0.01 Hz at 1000 Hz. Order 10
    # follows upstream modules.utils.bandpass_data; no spatial grid conversion.
    sos = signal.butter(10, [low, high], btype='bandpass', fs=fs, output='sos')
    data = np.empty((rows, samples), dtype=np.float32)
    for first in range(channel_from, channel_to + 1, 8):
        check_cancel(cancel)
        last = min(first + 7, channel_to)
        block = (read_group_window(group, start, end, first, last) if raw_data is None else
                 np.array(raw_data[first - 1:last, start:end], copy=True))
        if not np.isfinite(block).all():
            raise ValueError('DAS 数据含有 NaN 或无穷值，无法跟踪')
        block = signal.sosfiltfilt(sos, np.asarray(block, dtype=np.float64), axis=1)
        check_cancel(cancel)
        center = np.median(block, axis=1, keepdims=True)
        scale = 1.4826 * np.median(np.abs(block - center), axis=1, keepdims=True)
        valid = scale[:, 0] > 1e-12
        block = (block - center) / np.maximum(scale, 1e-12)
        block[~valid] = 0
        data[first - channel_from:last - channel_from + 1] = block
        progress(int(35 * (last - channel_from + 1) / rows), f'按 {fs:g} Hz 读取、滤波和归一化')
    result = detect_das_veh_tracks(data, fs, options, sample_origin=start, channel_origin=channel_from,
                                   sample_bounds=(visible_start, visible_end), cancel=cancel, progress=progress)
    return PrescreenResult(result.proposals, tuple(notes) + result.notes)


## B. 数据与参数配置

`SOURCE_PATHS` 填服务器上的真实路径，按采集先后排列；头部必须与软件支持的 BIN 格式一致。
窗口时间是**从整组数据起点开始的秒数**，不是墙上时钟，也不是 Notebook 运行时间。
默认查看 0–240 秒；若要复现之前的 190–430 秒范围，修改两个时间变量。
`channel_spacing=4.0` 是当前默认值，请核实实际每通道对应的距离。它直接影响投影速度换算。

0.01 Hz 的当前实现会尽可能取窗口两侧各 100 秒补充滤波，且实际处理长度至少 200 秒。
只有 240 秒数据时仍可处理，但会返回边界提示。采样率从 BIN 头读取并核对为 1000 Hz。
速度内部单位是 **m/s**；下面保留显式 `/ 3.6` 换算。

In [ ]:
SOURCE_PATHS = [
    # "/data/das/20260905_134433_tp00030.bin",
    # "/data/das/next_file.bin",
]
EXPECTED_SAMPLING_RATE = 1000.0
WINDOW_START_SECONDS = 0.0
WINDOW_END_SECONDS = 240.0

PARAMETERS = dict(adapter.DEFAULT_DAS_VEH_PARAMETERS)
PARAMETERS.update(
    channel_from=290, channel_to=520,
    frequency_low=0.01, frequency_high=1.0,
    channel_spacing=4.0,
    minimum_speed=0.0 / 3.6, maximum_speed=120.0 / 3.6,
    peak_prominence=1.5, minimum_separation=1.0,
    prominence_window=12.0, seed_sigma=0.08,
    seed_width=3, seed_stride=32,
    maximum_seeds=1000, maximum_candidates=100,
    minimum_track_channels=20, maximum_missed_channels=5,
    tracking_tolerance=1.2, sigma_a=0.01,
    direction="auto", polarity="both",
)

# 诊断图范围只控制显示，不约束算法搜索。
INSPECT_CHANNEL = 381
INSPECT_START_SECONDS = 0.0
INSPECT_END_SECONDS = 240.0
# Pass 0 为默认的正峰/向通道增大；其余 pass 标签在运行后打印。
INSPECT_PASS_INDEX = 0
INSPECT_SEED_SECTION_INDEX = 0

RUN_SWEEP = False
EXPORT_RESULTS = False
EXPORT_DIRECTORY = Path("das_veh_parameter_results")

### 参数影响：先明确它在哪一步生效

| 参数 | 当前默认值 / 单位 | 代码中的实际作用 | 增大后常见影响 |
|---|---|---|---|
| `channel_from`, `channel_to` | 290–520，含两端 | 指定原始通道范围；不使用人工轨迹限制搜索 | 范围扩大，计算量和可见轨迹增加 |
| `frequency_low`, `frequency_high` | 0.01、1 Hz | 10 阶 Butterworth SOS 双向滤波 | 改变信号形状；需重新读取与滤波 |
| `channel_spacing` | 4 m/通道 | 建立空间轴、KF 矩阵与速度换算 | 同一时间/通道斜率换算出的速度增大 |
| `peak_prominence` | 1.5，MAD 归一化后的突出程度 | 各通道 `find_peaks` 的最小 prominence | 峰更少，弱峰更容易漏掉 |
| `minimum_separation` | 1 秒 | 单通道峰和综合起点峰的最小间距，换算为采样点 | 临近峰更容易合并，可能漏掉紧邻事件 |
| `prominence_window` | 12 秒，总窗口 | prominence 的局部评估窗口；1000 Hz 对应 12001 点 | 改变用于评估突出程度的背景范围；不是每 12 秒只取一个峰 |
| `seed_sigma` | 0.08 秒 | `likelihood_1d` 中 Gaussian 的标准差 | 得分更宽，近邻通道的错位峰更易合成起点，也可能混合相邻事件 |
| `seed_width` | 3 个通道 | 每个起点区间累加多少通道的 Gaussian 得分 | 更多通道参与，但不是要求所有通道一致 |
| `seed_stride` | 32 个通道 | 起点区间的空间间隔 | 起点区间更少；**实际跟踪仍逐个原始通道执行** |
| `maximum_seeds` | 1000 个 | 按极性、方向和起点区间分配跟踪预算 | 搜索更多起点，耗时增加；不是峰检测上限 |
| `tracking_tolerance` | 1.2 秒 | 相对 KF 预测时间的峰关联范围 | 更能容忍偏差，也更容易关联到别的车 |
| `maximum_missed_channels` | 5 个连续通道 | 连续没找到匹配峰超过此值就停用该轨迹 | 更容易跨越缺峰，也可能错误接续 |
| `sigma_a` | 0.01，过程噪声系数 | 控制 KF 的 Q；内部再乘 `(fs/50)**2` | 改变预测不确定度和更新响应；不是车速/加速度的直接物理阈值 |
| `minimum_track_channels` | 20 个有效观测通道 | 观测不足的轨迹被过滤；也影响起点可放置范围 | 短碎片减少，但真实短轨迹也可能丢失；不要求这 20 个通道全连续 |
| `minimum_speed`, `maximum_speed` | 0–33.333 m/s，即 0–120 km/h | 根据轨迹拟合斜率做后验筛选，并有局部高速/反向检查 | 扩大速度范围可保留更多轨迹；**不决定关联搜索窗口** |
| `direction` | `auto` | 双向，或 `increasing` / `decreasing` | 选择空间遍历方向；auto 同时运行两种方向 |
| `polarity` | `both` | 正峰和原波形的负峰分开运行 | both 可能覆盖更多信号，也产生更多需要去重的候选 |
| `maximum_candidates` | 100 条 | 去重后按范围/强度排序保留候选 | 允许输出更多候选；不减少之前的跟踪计算 |

**建议一次只改一项**：先看单通道峰，调 `peak_prominence`、`minimum_separation`、`prominence_window`；
再看起点，调 `seed_sigma`、`seed_width`；最后看原始跟踪是否断裂/串线，调 `tracking_tolerance`、
`maximum_missed_channels`、`sigma_a`，再用最短长度和速度条件筛选。
候选数增加并不能证明拾取准确率变好；这里没有用人工标注作为搜索或评分依据。

当前实现还固定了这些细节，它们不在 `PARAMETERS` 中：
- `channel_step=1`，`observed_only=True`，`initialize_slope=True`，`nearest_peak=True`。
- 观测方差 `R=(fs/50)**2`；1000 Hz 时为 400 采样点²，等效标准差 20 ms。
- 原版起点综合得分的 `height=max(score)*0`，没有另设多通道一致性门槛。
- 去重使用 0.15 秒接近阈值、70% 共享范围、80% 一致比例；见 `_is_duplicate`。
- 返回候选折线按 0.08 秒 / 0.75 通道尺度简化；C4 的原始观测图不经过该简化。
- `tracking.py` 独立调用时的默认值可能不同；本 Notebook 从完整适配器入口运行，与软件一致。

## C. 完整运行与中间结果

下面的封装调用复制的 `analyze_das_veh` / `detect_das_veh_tracks`，并暂时包裹原函数记录输入、起点和原始观测。
包裹函数始终返回原函数的结果，不修改观测、参数、预算或筛选。
同一数据/时间/通道/频带下再次运行可复用第一次**原函数实际生成的**滤波归一化数组；找峰与跟踪仍完整重算。
`FORCE_REFILTER=True` 可强制重走读取和滤波。更换频带、窗口、通道或文件元数据会自动失效。

In [ ]:
from contextlib import ExitStack
from unittest.mock import patch
from dataclasses import asdict

_preprocess_cache = None

def load_source_group(paths):
    if not paths:
        raise ValueError("请先填写 SOURCE_PATHS：服务器上按时间排序的真实 BIN 路径。")
    resolved = [str(Path(p).expanduser().resolve(strict=True)) for p in paths]
    headers = [bin_reader.read_bin_header(p) for p in resolved]
    channels, fs = headers[0][2:4]
    if any(h[2:4] != (channels, fs) for h in headers):
        raise ValueError("各 BIN 文件的通道数和采样率必须一致。")
    if EXPECTED_SAMPLING_RATE is not None and fs != EXPECTED_SAMPLING_RATE:
        raise ValueError(f"BIN 头采样率 {fs:g} Hz 与期望 {EXPECTED_SAMPLING_RATE:g} Hz 不符；不会改采样率。")
    return data_group.DataGroup.from_files(resolved, [h[1] for h in headers], channels, fs)

def run_experiment(parameters, *, force_refilter=False, verbose=True):
    global _preprocess_cache
    options = adapter._options(parameters)
    group = load_source_group(SOURCE_PATHS)
    fs = float(group.sampling_rate)
    first, last = int(round(WINDOW_START_SECONDS * fs)), int(round(WINDOW_END_SECONDS * fs))
    ch1, ch2 = int(options['channel_from']), int(options['channel_to'])
    # Validate even when reusing processed data; analysis bounds stay exact.
    if not 0 <= first < last <= group.total_samples:
        raise ValueError(f"窗口超出源范围 0–{group.total_samples / fs:g} 秒，请修改 WINDOW_*_SECONDS。")
    if ch1 < 1 or ch2 > group.channel_count or ch2 - ch1 + 1 < 8:
        raise ValueError("通道范围必须在源数据内，且至少包含 8 个通道。")
    signatures = tuple((s.path, Path(s.path).stat().st_size, Path(s.path).stat().st_mtime_ns)
                       for s in group.segments)
    key = (signatures, fs, first, last, ch1, ch2, options['frequency_low'], options['frequency_high'])
    cached = (_preprocess_cache is not None and _preprocess_cache['key'] == key and not force_refilter)
    record = {'parameters': dict(options), 'sampling_rate': fs,
              'sample_bounds': (first, last), 'channel_range': (ch1, ch2),
              'source_paths': [s.path for s in group.segments], 'passes': [],
              'cache_used': cached, 'filter_notes': ()}
    signs = [1, -1] if options['polarity'] == 'both' else ([1] if options['polarity'] == 'positive' else [-1])
    directions = [1, -1] if options['direction'] == 'auto' else ([1] if options['direction'] == 'increasing' else [-1])
    tracker_passes = {}
    original_detect = adapter.detect_das_veh_tracks
    original_seeds = tracking.KF_tracking.detect_in_one_section
    original_tracks = tracking.KF_tracking.tracking_with_veh_base
    last_print = [0.0]

    def progress(percent, message):
        now = time.perf_counter()
        if verbose and (now - last_print[0] >= 3 or percent == 100):
            print(f"{percent:3d}% {message}", flush=True)
            last_print[0] = now

    def capture_preprocessing(data, sampling_rate, parameters=None, **kwargs):
        record['processed_data'] = data
        record['sample_origin'] = int(kwargs.get('sample_origin', 0))
        record['channel_origin'] = int(kwargs.get('channel_origin', 1))
        record['detect_kwargs'] = {k: v for k, v in kwargs.items() if k not in ('cancel', 'progress')}
        assert sampling_rate == fs
        detected = original_detect(data, sampling_rate, parameters, **kwargs)
        record['detect_notes'] = detected.notes
        return detected

    def pass_for(tracker):
        if tracker not in tracker_passes:
            index = len(record['passes'])
            item = {'sign': signs[index // len(directions)],
                    'direction': directions[index % len(directions)],
                    'peak_locations': tracker.args['peak_locations'],
                    'arguments': dict(tracker.args), 'sections': [], 'raw_tracks': []}
            tracker_passes[tracker] = item
            record['passes'].append(item)
        return tracker_passes[tracker]

    def capture_seeds(tracker, *args, **kwargs):
        bases = original_seeds(tracker, *args, **kwargs)
        start_x = args[0] if args else kwargs['start_x']
        start_row = int(np.argmin(np.abs(start_x - tracker.x_axis)))
        pass_for(tracker)['sections'].append({'start_row': start_row,
                                              'all_seed_samples': bases.copy()})
        return bases

    def capture_tracks(tracker, *args, **kwargs):
        states = original_tracks(tracker, *args, **kwargs)
        start_x = args[0] if args else kwargs['start_x']
        bases = args[2] if len(args) > 2 else kwargs['veh_base']
        pass_for(tracker)['raw_tracks'].append({
            'start_row': int(np.argmin(np.abs(start_x - tracker.x_axis))),
            'used_seed_samples': np.asarray(bases).copy(), 'states': states.copy()})
        return states

    started = time.perf_counter()
    with ExitStack() as stack:
        stack.enter_context(patch.object(adapter, 'detect_das_veh_tracks', capture_preprocessing))
        stack.enter_context(patch.object(tracking.KF_tracking, 'detect_in_one_section', capture_seeds))
        stack.enter_context(patch.object(tracking.KF_tracking, 'tracking_with_veh_base', capture_tracks))
        if cached:
            snapshot = _preprocess_cache
            result = capture_preprocessing(snapshot['data'], fs, options,
                                           **snapshot['kwargs'], progress=progress)
            record['filter_notes'] = snapshot['filter_notes']
            result = prescreen.PrescreenResult(result.proposals, record['filter_notes'] + result.notes)
        else:
            result = adapter.analyze_das_veh(group, None, first, last, ch1, ch2, options, progress=progress)
            # Retain the exact prefix added by analyze_das_veh.
            prefix_count = len(result.notes) - len(record['detect_notes'])
            record['filter_notes'] = tuple(result.notes[:prefix_count])
            _preprocess_cache = {'key': key, 'data': record['processed_data'],
                                 'kwargs': dict(record['detect_kwargs']), 'filter_notes': record['filter_notes']}
    record['result'] = result
    record['elapsed_seconds'] = time.perf_counter() - started
    counts = [np.sum(np.isfinite(job['states']) &
                     (job['states'] + record['sample_origin'] >= first) &
                     (job['states'] + record['sample_origin'] < last), axis=1)
              for p in record['passes'] for job in p['raw_tracks']]
    record['raw_observation_counts'] = np.concatenate(counts).astype(int) if counts else np.array([], dtype=int)
    if verbose:
        print(f"处理数组 {record['processed_data'].shape}；采样率 {fs:g} Hz；"
              f"候选 {len(result.proposals)} 条；耗时 {record['elapsed_seconds']:.2f} s；滤波缓存={cached}")
        for note in result.notes:
            print("提示：", note)
    return record

**C1. 运行一次。** 改了配置后重跑本单元。第一次会读取、滤波并执行全部跟踪。

In [ ]:
FORCE_REFILTER = False
run = run_experiment(PARAMETERS, force_refilter=FORCE_REFILTER)
print("当前参数：")
print(json.dumps(run['parameters'], ensure_ascii=False, indent=2))
for index, item in enumerate(run['passes']):
    visible_peaks = sum(np.count_nonzero(
        (locations + run['sample_origin'] >= run['sample_bounds'][0]) &
        (locations + run['sample_origin'] < run['sample_bounds'][1]))
        for locations in item['peak_locations'])
    all_seeds = sum(len(s['all_seed_samples']) for s in item['sections'])
    used_seeds = sum(len(job['used_seed_samples']) for job in item['raw_tracks'])
    print(f"Pass {index}: polarity={item['sign']:+d}, direction={item['direction']:+d}, "
          f"窗口内各通道峰总数={visible_peaks}, 未截窗口/预算的综合起点={all_seeds}, "
          f"实际跟踪起点={used_seeds}")

**C2. 单通道峰：回答“这个通道到底挑出了哪些峰”。**

直接读取本次适配器缓存的 `peak_locations`；没有再发明一套检测规则。
正峰和负峰属于两个独立的检测 pass，所以两者之间不要求满足 1 秒间隔。
诊断显示范围不改变检测窗口。修改 `INSPECT_CHANNEL` 和 `INSPECT_*_SECONDS` 后重跑本单元即可。

In [ ]:
def inspection_slice(record, start_seconds, end_seconds):
    fs = record['sampling_rate']
    first, last = record['sample_bounds']
    start = max(first, int(round(start_seconds * fs)))
    end = min(last, int(round(end_seconds * fs)))
    if end <= start:
        raise ValueError("诊断显示范围与当前拾取窗口没有重叠。")
    offset = record['sample_origin']
    return slice(start - offset, end - offset), (start / fs, end / fs)

def plot_channel_peaks(record, channel, start_seconds, end_seconds):
    row = int(channel) - record['channel_origin']
    data = record['processed_data']
    if not 0 <= row < data.shape[0]:
        raise ValueError("INSPECT_CHANNEL 不在本次处理通道内。")
    view, bounds = inspection_slice(record, start_seconds, end_seconds)
    fs, origin = record['sampling_rate'], record['sample_origin']
    time_axis = (origin + np.arange(view.start, view.stop)) / fs
    trace = data[row]
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.plot(time_axis, trace[view], color='0.35', linewidth=0.7, label='Filtered + MAD normalized')
    seen_signs = set()
    for item in record['passes']:
        if item['sign'] in seen_signs:
            continue
        seen_signs.add(item['sign'])
        oriented_row = row if item['direction'] > 0 else data.shape[0] - 1 - row
        locations = item['peak_locations'][oriented_row]
        locations = locations[(locations >= view.start) & (locations < view.stop)]
        symbol, color = ('^', 'tab:red') if item['sign'] > 0 else ('v', 'tab:blue')
        ax.scatter((origin + locations) / fs, trace[locations], marker=symbol, s=32,
                   color=color, label=f"Polarity {item['sign']:+d}: {len(locations)} peaks")
        print(f"通道 {channel}，极性 {item['sign']:+d}，显示范围内 {len(locations)} 个峰；"
              f"前 20 个时间(s)：{np.round((origin + locations[:20]) / fs, 3).tolist()}")
    ax.set(xlim=bounds, xlabel='Seconds from source start', ylabel='Normalized response',
           title=f'Channel {channel}: peaks from the actual detector cache')
    ax.legend()
    fig.tight_layout()
    plt.show()
    return fig

channel_figure = plot_channel_peaks(run, INSPECT_CHANNEL, INSPECT_START_SECONDS, INSPECT_END_SECONDS)

**C3. 起点综合得分：查看 `likelihood_1d` 和第 49 行选出了什么。**

从本次实际 pass 和起点区间重建 `KF_tracking`，再次调用原 `detect_in_one_section`。
记录原函数调用的每条 Gaussian 得分再相加用于画图，并核对起点与本次运行完全相同。
图中的起点是截取时间窗口和预算之前的结果；实际进入跟踪的起点另用绿色标记。

In [ ]:
def plot_seed_score(record, pass_index, section_index, start_seconds, end_seconds):
    if not record['passes']:
        raise ValueError("没有可用起点区间；检查 minimum_track_channels 是否超过处理通道数。")
    item = record['passes'][pass_index]
    section = item['sections'][section_index]
    data = record['processed_data']
    oriented = data if item['direction'] > 0 else data[::-1]
    fs = record['sampling_rate']
    t_axis = np.arange(data.shape[1], dtype=float) / fs
    x_axis = np.arange(data.shape[0], dtype=float) * record['parameters']['channel_spacing']
    args = dict(item['arguments'])
    args['progress'] = lambda fraction: None
    tracker = tracking.KF_tracking(oriented, t_axis, x_axis, args)
    score = np.zeros(len(t_axis))
    original_likelihood = tracking.likelihood_1d

    def capture_likelihood(*positional, **keywords):
        contribution = original_likelihood(*positional, **keywords)
        score[:] += contribution
        return contribution

    start = section['start_row']
    with patch.object(tracking, 'likelihood_1d', capture_likelihood):
        seeds = tracker.detect_in_one_section(x_axis[start], nx=record['parameters']['seed_width'],
                                              sigma=record['parameters']['seed_sigma'], show_plot=False)
    np.testing.assert_array_equal(seeds, section['all_seed_samples'])
    jobs = [job for job in item['raw_tracks'] if job['start_row'] == start]
    used = np.concatenate([job['used_seed_samples'] for job in jobs]) if jobs else np.array([], dtype=int)
    view, bounds = inspection_slice(record, start_seconds, end_seconds)
    origin = record['sample_origin']
    visible = seeds[(seeds >= view.start) & (seeds < view.stop)]
    used_visible = used[(used >= view.start) & (used < view.stop)]
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.plot((origin + np.arange(view.start, view.stop)) / fs, score[view], linewidth=0.8)
    ax.scatter((origin + visible) / fs, score[visible], marker='x', color='tab:orange',
               label=f'All detected seeds in display range: {len(visible)}')
    ax.scatter((origin + used_visible) / fs, score[used_visible], s=45, facecolors='none',
               edgecolors='tab:green', label=f'Actually tracked seeds in display range: {len(used_visible)}')
    channel = record['channel_origin'] + (start if item['direction'] > 0 else data.shape[0] - 1 - start)
    ax.set(xlim=bounds, xlabel='Seconds from source start', ylabel='Gaussian sum (not probability)',
           title=f"Pass {pass_index}, section {section_index}, first original channel {channel}, "
                 f"sigma={record['parameters']['seed_sigma']:g} s")
    ax.legend()
    fig.tight_layout()
    plt.show()
    return fig

seed_figure = plot_seed_score(run, INSPECT_PASS_INDEX, INSPECT_SEED_SECTION_INDEX,
                             INSPECT_START_SECONDS, INSPECT_END_SECONDS)

**C4. 原始跟踪 vs 最终候选。**

左图：原 `tracking_with_veh_base` 返回的实际匹配观测，尚未做最短长度、速度、去重和输出简化。
灰色点包括失败的短轨迹；彩色折线仅连接达到当前最短长度的原始结果，断点不填补。
右图：软件当前入口最终返回的候选折线。左图通过、右图消失时，再检查后验筛选与去重。
折线穿过缺峰处只是连线显示，不代表新增了观测点。全部候选都是待人工检查的初步拾取。

In [ ]:
def raw_observed_points(record, item, job, state):
    rows = np.flatnonzero(np.isfinite(state)) + job['start_row']
    samples = np.rint(record['sample_origin'] + state[np.isfinite(state)]).astype(np.int64)
    first, last = record['sample_bounds']
    keep = (samples >= first) & (samples < last)
    rows, samples = rows[keep], samples[keep]
    channels = (rows if item['direction'] > 0 else record['processed_data'].shape[0] - 1 - rows)
    channels = channels + record['channel_origin']
    return np.column_stack((samples, channels))

def draw_background(ax, record, start_seconds, end_seconds):
    view, bounds = inspection_slice(record, start_seconds, end_seconds)
    image = record['processed_data'][:, view]
    vmax = float(np.percentile(np.abs(image), 98))
    vmax = max(vmax, 1e-12)
    ax.imshow(image, origin='lower', aspect='auto', cmap='gray', vmin=-vmax, vmax=vmax,
              extent=[*bounds, record['channel_range'][0] - 0.5, record['channel_range'][1] + 0.5])
    ax.set(xlim=bounds, xlabel='Seconds from source start', ylabel='Original channel')

def plot_tracks(record, start_seconds, end_seconds):
    fig, axes = plt.subplots(1, 2, figsize=(20, 8), sharex=True, sharey=True)
    for ax in axes:
        draw_background(ax, record, start_seconds, end_seconds)
    fs = record['sampling_rate']
    raw_count = 0
    retained_length_count = 0
    for item in record['passes']:
        for job in item['raw_tracks']:
            for state in job['states']:
                points = raw_observed_points(record, item, job, state)
                if not len(points):
                    continue
                raw_count += 1
                if len(points) < record['parameters']['minimum_track_channels']:
                    axes[0].scatter(points[:, 0] / fs, points[:, 1], color='tab:gray', s=3, alpha=0.3)
                else:
                    retained_length_count += 1
                    axes[0].plot(points[:, 0] / fs, points[:, 1], '.-', markersize=1.5, linewidth=0.6, alpha=0.6)
    for proposal in record['result'].proposals:
        points = np.asarray(proposal.vertices)
        axes[1].plot(points[:, 0] / fs, points[:, 1], linewidth=1.1)
        axes[1].annotate(str(proposal.identifier), (points[0, 0] / fs, points[0, 1]), fontsize=8)
    axes[0].set_title(f'Raw observations: {raw_count} paths; {retained_length_count} meet length only')
    axes[1].set_title(f'Final candidates: {len(record["result"].proposals)}; {record["elapsed_seconds"]:.2f} s')
    fig.tight_layout()
    plt.show()
    return fig

track_figure = plot_tracks(run, INSPECT_START_SECONDS, INSPECT_END_SECONDS)

print("id | channels | time(s) | continuity | mean prominence | ambiguous | simplified nodes")
for proposal in run['result'].proposals:
    points = np.asarray(proposal.vertices)
    print(f"{proposal.identifier:2d} | {points[:,1].min():g}–{points[:,1].max():g} | "
          f"{points[:,0].min()/run['sampling_rate']:.3f}–{points[:,0].max()/run['sampling_rate']:.3f} | "
          f"{proposal.continuity:.3f} | {proposal.strength:.3f} | {proposal.ambiguous} | {len(points)}")
# continuity = 实际观测通道数 / 覆盖的通道跨度；不是识别置信概率。
# ambiguous = 时间对线性拟合的抖动超过 tracking_tolerance；不是已确认的交叉识别。

## D. 单因素批量对比

默认不执行。将 `RUN_SWEEP=True` 后运行下面两格。
每项都以当前 `PARAMETERS` 为基准，只修改 `changes`，通过同一个原函数入口执行。
数据与频带相同时复用原入口生成的滤波结果，不缓存上次的峰或 KF 结果。
曲线数和耗时只是诊断，不能替代正确率；批量实验也不读取人工标注。

In [ ]:
TRIALS = [
    ("baseline", {}),
    ("prominence=1.0", {"peak_prominence": 1.0}),
    ("prominence=2.0", {"peak_prominence": 2.0}),
    ("tolerance=0.6s", {"tracking_tolerance": 0.6}),
    ("tolerance=1.8s", {"tracking_tolerance": 1.8}),
    ("min_channels=40", {"minimum_track_channels": 40}),
    ("sigma_a=0.003", {"sigma_a": 0.003}),
    # ("seed_sigma=0.16s", {"seed_sigma": 0.16}),
    # ("separation=0.5s", {"minimum_separation": 0.5}),
    # ("prominence_window=24s", {"prominence_window": 24.0}),
]

experiments = []
if RUN_SWEEP:
    for label, changes in TRIALS:
        print(f"\n--- {label} ---")
        trial_parameters = {**PARAMETERS, **changes}
        trial_run = run_experiment(trial_parameters, verbose=False)
        experiments.append((label, trial_run))
        valid = trial_run['raw_observation_counts']
        median = float(np.median(valid)) if len(valid) else 0.0
        print(f"候选={len(trial_run['result'].proposals)}, 原始路径={len(valid)}, "
              f"原始观测数中位数={median:g}, 耗时={trial_run['elapsed_seconds']:.2f}s, "
              f"缓存={trial_run['cache_used']}")
        for note in trial_run['result'].notes:
            print("提示：", note)
else:
    print("批量实验未开启；将 RUN_SWEEP 改为 True 后运行本单元。")

In [ ]:
if experiments:
    ncols = 2
    nrows = int(np.ceil(len(experiments) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows), squeeze=False, sharex=True, sharey=True)
    for ax, (label, trial) in zip(axes.flat, experiments):
        draw_background(ax, trial, INSPECT_START_SECONDS, INSPECT_END_SECONDS)
        for proposal in trial['result'].proposals:
            points = np.asarray(proposal.vertices)
            ax.plot(points[:, 0] / trial['sampling_rate'], points[:, 1], linewidth=0.8)
        ax.set_title(f"{label}: {len(trial['result'].proposals)} candidates")
    for ax in list(axes.flat)[len(experiments):]:
        ax.set_visible(False)
    fig.tight_layout()
    plt.show()

## E. 可选导出参数和拾取结果

默认关闭。`EXPORT_RESULTS=True` 时，把参数、候选、源码来源/校验、原始观测状态和诊断图写入指定目录。
导出的是你主动保存的实验结果；不会生成临时 Python 源文件或修改输入 BIN。
`sample` 始终是整组源数据中的原始采样点编号，`channel` 是原始通道编号。
原始状态数组中的采样点相对 `sample_origin`，通道列相对对应 `start_row` 和 pass 方向。

In [ ]:
if EXPORT_RESULTS:
    EXPORT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    report = {
        'parameters': run['parameters'], 'sampling_rate': run['sampling_rate'],
        'sample_origin': run['sample_origin'], 'channel_origin': run['channel_origin'],
        'sample_bounds': run['sample_bounds'], 'channel_range': run['channel_range'],
        'source_paths': run['source_paths'], 'processed_shape': list(run['processed_data'].shape),
        'elapsed_seconds': run['elapsed_seconds'], 'cache_used': run['cache_used'],
        'notes': list(run['result'].notes), 'source_manifest': SOURCE_MANIFEST,
        'versions': RUNTIME_VERSIONS, 'manual_annotation_constraints': False,
        'proposals': [asdict(p) for p in run['result'].proposals],
        'raw_jobs': [],
        'sweep': [{'label': label, 'parameters': item['parameters'],
                   'candidate_count': len(item['result'].proposals),
                   'elapsed_seconds': item['elapsed_seconds'], 'notes': list(item['result'].notes),
                   'proposals': [asdict(p) for p in item['result'].proposals]}
                  for label, item in experiments],
    }
    arrays = {}
    for pass_index, item in enumerate(run['passes']):
        for job_index, job in enumerate(item['raw_tracks']):
            key = f'pass_{pass_index}_job_{job_index}'
            arrays[key + '_states'] = job['states']
            arrays[key + '_seeds'] = job['used_seed_samples']
            report['raw_jobs'].append({'key': key, 'sign': item['sign'],
                                        'direction': item['direction'], 'start_row': job['start_row']})
    (EXPORT_DIRECTORY / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    np.savez_compressed(EXPORT_DIRECTORY / 'raw_observed_states.npz', **arrays)
    for name, figure in [('channel_peaks', channel_figure), ('seed_score', seed_figure), ('tracks', track_figure)]:
        figure.savefig(EXPORT_DIRECTORY / f'{name}.png', dpi=150)
    print("已导出：", EXPORT_DIRECTORY.resolve())
else:
    print("导出未开启，未写出实验结果文件。")

## F. 如何判断下一步该调哪个参数

- 单通道上目标位置根本没有峰：先检查滤波后的波形，再看突出程度、最小间隔和 prominence 窗口。
- 有峰但综合起点没有：看起点区间有没有覆盖车辆可见段，检查 `seed_sigma` / `seed_width` / `seed_stride`。
- 起点有，但绿色“实际跟踪”标记少：窗口外起点被过滤，或起点预算不足；查看运行提示。
- 原始轨迹很短：看时间容差是否太紧、连续缺峰是否过多、相邻通道的峰是否来自同一事件。
- 原始轨迹串到另一条线：结合单通道候选和起点，尝试较小的时间容差；再对比过程噪声。
- 原始轨迹足够长，最终候选却没了：看斜率符号、投影速度范围、局部 8 间隔斜率检查和去重逻辑。
- 只有起点区间间隔较大的问题：减小 `seed_stride` 会增加起点位置；它从不把实际跟踪通道降采样。
- 运行提示预算上限：优先确认预算是否影响比较，必要时增加 `maximum_seeds`，并保留该次参数记录。

核心状态是“采样点时间位置 + 每米变化的采样点数”，不是直接以 km/h 表示的速度。
调参应对照信号和轨迹形状，固定数据、窗口与其他参数；候选是初步拾取结果。

## G. 源码来源与许可证

# das_veh source provenance and adaptation

Source: https://github.com/syyuan93/das_veh

Pinned commit: `b1831d2d2a5636ec416095f9647a33881afaeed1` (2023-11-03).

Vendored files:

- `apis/tracking.py` → `tracking.py`
- `modules/car_tracking_utils.py` → `car_tracking_utils.py`

License: MIT, Copyright (c) 2023 Siyuan Yuan. Full notice is in
`licenses/das_veh-LICENSE.txt`. Upstream notes that Kalman code is based on
https://github.com/jingxiaoliu/TelecomTM; that attribution is preserved.

## Changes to upstream source

- Remove the author's absolute `sys.path`; use a package-relative import.
- Remove an unused ObsPy import. No additional runtime dependencies.
- Clamp the seed section at the last channel; allow empty results and empty
  interpolation rows. Avoid computing plot percentiles when plots are disabled.
- Convert single-element NumPy arrays to scalars for NumPy compatibility;
  fix the tuple indexing in the nearest/positive peak selection.
- Fix the returned channel count when it is not divisible by the spatial step.
- Optional adapter arguments: cached peak locations, cancellation/progress,
  spatial step, observation covariance, association window, nearest residual,
  consecutive missing-channel limit, two-point slope initialization, and raw
  observed states before the upstream extrapolation/cleanup.
- Keep the original transition matrix A, process matrix Q, Kalman gain and
  covariance update equations. Adapter noise variances scale by `(fs/50)^2`
  because the upstream state uses temporal sample indices, not seconds.
- For long uniformly sampled time axes, compute the original Gaussian seed
  likelihood by FFT convolution at **every original sample**, truncating the
  Gaussian at eight standard deviations and suppressing numerical round-off
  seed peaks. The direct Gaussian implementation remains for shorter inputs.

`utils/classes/das_veh_tracking.py` is the separate application adapter:

- BIN windows and filter padding at the actual acquisition rate; no temporal
  resampling/decimation or spatial interpolation.
- Upstream order-10 temporal Butterworth bandpass, stable float64 SOS, followed
  by per-channel MAD normalization so prominence controls are comparable.
- Forward/reversed original-channel views and separate positive/negative peaks.
- Return measured sample/channel coordinates; no endpoint extrapolation.
- Original-channel consecutive-gap stopping, projected speed validation,
  substantial-overlap de-duplication, and polyline simplification for editing.
- 0 m/s lower speed bound means disabled lower-speed filtering; no division by
  zero. Existing annotations are not an input and never constrain detection.
- Peak arrays are cached; seeds are budgeted across time/space/direction/polarity.

The original spatial resampling, spatial bandpass, absolute amplitude/noise
thresholds, surface-wave window extraction, and dispersion imaging workflow
are not imported into the viewer. Their site assumptions do not describe this
BIN geometry. The separate existing beam backend retains its original behavior.

## Validation

Run `test/das_veh_tracking_check.py` for full-rate, polarity/direction, empty
results, missing channels, measured endpoints, static/noise rejection, projected
speed, likelihood equivalence, cross-file coordinates, cancellation and review.
`test/das_veh_real_check.py` reads only `das_context` from a local source project;
it does not use its annotation geometries or labels.


```text
MIT License

Copyright (c) 2023 Siyuan Yuan

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.


```